In [1]:
import pandas as pd
import os
import pickle
import bmra_prep
import bmra_prep.pathway_activity.prediction

In [2]:
cell_line ='BC3C'

data_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}_oct_aug/00_outputs_2020_{cell_line}/"
out_dir = f"/home/jing/Phd_project/project_UCD_blca/blca_publication_OUTPUT/blca_publication_OUTPUT_bmra/blca_publication_OUTPUT_bmra_{cell_line}_oct_aug/01_outputs_2020_{cell_line}/"


os.makedirs(out_dir, exist_ok = True)

# Load Data

In [3]:
# load metdadata dict and extract used elements
with open(os.path.join(data_dir, "metadata.pickle"), "rb") as f:
    all_metadata = pickle.load(f)

n_modules = all_metadata["n_modules"]
n_genes = all_metadata["n_genes"]
n_experiments = all_metadata["n_experiments"]

modules = all_metadata["modules"]
exp_ids = all_metadata["exp_ids"]
genes = all_metadata["genes"]

In [4]:
# load data
L1000_df = pd.read_csv(
    os.path.join(data_dir, "L1000_Data_norm_data.csv"),
    index_col = 0,
)

x = L1000_df.values
x.shape

(978, 144)

In [5]:
# load doses and perturbation matrix
inhib_conc_matrix = pd.read_csv(
    os.path.join(data_dir, "inhib_conc_annotated.csv"),
    index_col = 0,
).values

ic50_matrix = pd.read_csv(
    os.path.join(data_dir, "ic50_annotated.csv"),
    index_col = 0,
).values

# gamma_matrix = pd.read_csv(
#     os.path.join(data_dir, "gamma_annotated.csv"),
#     index_col = 0,
# ).values

pert_matrix = pd.read_csv(
    os.path.join(data_dir, "pert_annotated.csv"),
    index_col = 0,
).values

In [6]:
# y_true = (1 + gamma_matrix * inhib_conc_matrix / ic50_matrix) / (1 + inhib_conc_matrix / ic50_matrix)

y_true = 1 / (1 + inhib_conc_matrix / ic50_matrix)

display(y_true.shape)
y_true

(12, 144)

array([[1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       ...,
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 1.        , 1.        ,
        1.        ],
       [1.        , 1.        , 1.        , ..., 0.41176471, 0.41176471,
        0.41176471]])

## Run models

In [7]:
a_coeffs = bmra_prep.pathway_activity.prediction.predict_coeffs(
    x, y_true, pert_matrix, 200_000, 10, 10, 10, 100)

In [8]:
a_coeffs_df = pd.DataFrame(a_coeffs, index = modules, columns = genes)
a_coeffs_df.to_csv(os.path.join(out_dir, "a_coeffs.csv"))
#a_coeffs_df = pd.read_csv(os.path.join(out_dir,'a_coeffs.csv'),index_col=0)
#a_coeffs = a_coeffs_df.values
display(a_coeffs_df.astype(bool).sum(axis='columns'))
display(a_coeffs_df)

Androgen    978
CDK1_2      978
CDK4_6      978
EGFR        978
Estrogen    978
FGFR        978
PI3K        978
p53         978
TOP2A       978
Src         978
TGFb        978
SMAD3       978
dtype: int64

,AARS,ABCB6,ABCC5,ABCF1,ABCF3,ABHD4,ABHD6,ABL1,ACAA1,ACAT2,...,ZMIZ1,ZMYM2,ZNF131,ZNF274,ZNF318,ZNF395,ZNF451,ZNF586,ZNF589,ZW10
Androgen,-0.000014,9.652995e-06,3.327697e-06,0.000006,0.000003,-9.494902e-06,-0.000010,-1.044053e-05,-1.575184e-05,-8.781003e-06,...,-3.552570e-07,-1.050276e-05,-0.000035,-0.000021,-8.314237e-06,0.000023,-0.000017,3.249690e-06,0.000016,2.399911e-05
CDK1_2,0.000013,-5.296721e-06,-7.971890e-06,-0.000019,-0.000003,3.609562e-05,-0.000004,1.471215e-05,1.739837e-05,-7.194958e-05,...,-2.766573e-05,-2.744540e-06,0.000013,-0.000017,-2.046115e-05,-0.000003,0.000005,-2.548399e-05,-0.000007,-2.007420e-05
CDK4_6,-0.000004,-9.882096e-06,-2.325678e-05,-0.000003,-0.000012,1.805978e-05,-0.000003,-1.427514e-05,2.752391e-05,4.468570e-06,...,1.258304e-05,-1.634604e-05,0.000028,-0.000025,1.759946e-05,0.000005,0.000026,1.476869e-05,0.000022,-2.029013e-06
EGFR,-0.000065,-2.475034e-05,2.046072e-06,-0.000013,0.000016,-4.029908e-04,0.000020,-3.150991e-06,-6.279721e-06,-2.660252e-02,...,-1.583476e-05,9.214680e-06,-0.000028,0.000013,-5.499083e-06,0.000305,-0.000012,-2.394036e-06,0.000036,-3.012223e-05
Estrogen,-0.000014,-1.202734e-05,5.397827e-08,0.000014,0.000041,3.541107e-05,0.000003,-3.511039e-06,5.862565e-06,-2.729085e-01,...,1.466929e-05,1.287413e-05,-0.000018,0.000023,3.983460e-06,-0.000047,-0.000008,-2.740863e-06,0.000007,-6.833570e-06
FGFR,-0.000569,2.473278e-05,7.904814e-06,0.000021,-0.000015,-1.367402e-05,-0.000013,-2.727425e-06,1.288582e-05,-2.314134e-04,...,4.226485e-05,-1.123992e-05,0.000009,-0.000005,1.239996e-05,0.000019,-0.000009,1.012105e-05,-0.000003,9.849653e-06
PI3K,0.000003,2.210633e-07,3.041134e-06,-0.000002,0.000022,-1.513043e-05,0.000011,1.458001e-05,-7.965464e-06,-3.532972e-05,...,1.776684e-06,2.768907e-08,0.000013,-0.000002,-3.373568e-07,-0.000132,-0.000022,-6.355774e-06,0.000038,2.566903e-02
p53,0.000003,-1.851743e-05,-1.441352e-05,0.000023,0.000013,-2.006636e-05,0.231212,-1.010414e-06,6.623509e-05,1.739650e-06,...,1.718086e-05,-1.909595e-05,-0.000026,-0.000007,5.140605e-05,0.000006,0.000029,-2.244037e-05,-0.000010,1.089434e-05
TOP2A,-0.000014,3.995766e-06,-4.123334e-05,0.000032,-0.000009,3.123014e-07,-0.000021,-2.422583e-05,1.210762e-05,-1.015747e-05,...,-1.745728e-05,2.506735e-05,-0.000002,0.000020,1.746295e-05,0.000016,0.000013,-1.224467e-05,0.000025,-3.890693e-05
Src,-0.000022,1.990872e-05,-9.913872e-06,-0.000001,0.000025,1.006953e-05,0.000042,2.739477e-05,-4.980407e-06,9.113069e-07,...,2.828184e-05,2.650954e-05,-0.000014,0.000010,-5.672762e-06,-0.000021,-0.000013,3.756874e-07,0.000037,8.841487e-07


In [9]:
#pathway_activity = a_coeffs @ x
#pathway_activity.shape

In [10]:
R_global = bmra_prep.pathway_activity.calc_global_response_from_pathway_activity(
    bmra_prep.pathway_activity.calc_pathway_activity(x,a_coeffs),
    modules,
    L1000_df.columns
)
R_global_df = R_global.dataframe
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD3in_V11,SMAD3in_V12,SMAD3in_V13,SMAD3in_V14,SMAD3in_V15,SMAD3in_V16,SMAD3in_V17,SMAD3in_V18,SMAD3in_V19,SMAD3in_V20
Androgen,-0.052095,-0.009089,0.002049,-0.039778,0.001388,0.020524,0.004034,0.016122,0.021111,0.035960,...,0.012764,0.004873,-0.002185,0.001210,0.000265,0.007804,-0.004034,0.008464,0.003308,0.021534
CDK1_2,-0.768273,-0.614755,0.075325,-0.079904,0.089680,-0.282097,0.017335,-0.287704,0.120143,0.055118,...,-0.480374,-0.437256,-0.400662,-0.417611,-0.386523,-0.403037,-0.394879,-0.437924,-0.435277,-0.472746
CDK4_6,-0.092411,-0.210950,-0.265290,-0.141717,-0.163803,-0.103093,-0.016878,-0.028821,-0.218272,-0.010762,...,0.198343,0.189244,0.177556,0.140909,0.138950,0.150663,0.172020,0.164796,0.190069,0.163447
EGFR,0.598441,0.498146,0.228981,0.371224,0.486402,0.113873,-0.415507,0.263087,0.286899,-0.047077,...,-1.225406,-0.821121,-0.806630,-0.706686,-0.677071,-0.692529,-0.744456,-0.899220,-0.879137,-0.933293
Estrogen,-0.125409,-0.208362,-0.209045,-0.410515,-0.943351,-0.311418,-0.084928,-0.236092,-0.162215,-0.039650,...,-0.296804,-0.265812,-0.238593,-0.263146,-0.243248,-0.251675,-0.229277,-0.276421,-0.260162,-0.304476
FGFR,-0.110612,-0.176118,-0.089455,0.051478,-0.028281,-0.407428,-0.033354,-0.026855,-0.061340,-0.301084,...,-0.854376,-0.700109,-0.638602,-0.638719,-0.569303,-0.617494,-0.624983,-0.711393,-0.717033,-0.792202
PI3K,-1.910966,-1.699159,-1.425575,-1.241725,-0.701968,0.295109,-0.143859,-0.177113,-0.848384,-0.273685,...,-0.177298,-0.139084,-0.120742,-0.114059,-0.099701,-0.140369,-0.131804,-0.135862,-0.155846,-0.180301
p53,-0.210364,-0.217898,-0.125995,-0.404942,0.049696,-1.630029,-1.476804,-0.119178,-0.087987,-1.326474,...,0.016668,-0.000954,0.011649,0.021806,-0.006903,0.014973,0.042119,-0.003211,0.037732,0.001231
TOP2A,-0.194672,0.095617,-0.233097,-0.163237,-0.127492,0.043017,0.079112,-2.000421,-0.196845,-0.298854,...,0.108585,0.092367,0.083997,0.071913,0.067490,0.079518,0.083257,0.086786,0.095575,0.097503
Src,-0.928588,-1.677513,0.522790,-1.205287,0.571455,-1.127273,0.495578,0.394501,0.397006,0.447192,...,-0.127760,-0.154060,-0.098183,-0.127832,-0.056520,-0.117954,-0.098885,-0.113815,-0.155771,-0.166602


In [11]:
R_global_df.to_csv(os.path.join(out_dir, "R_global_annotated.csv"))
display(R_global_df)

,ASG002_BC3C_24H:A10,ASG002_BC3C_24H:A11,ASG002_BC3C_24H:A19,ASG002_BC3C_24H:A20,ASG002_BC3C_24H:A21,ASG002_BC3C_24H:B10,ASG002_BC3C_24H:B11,ASG002_BC3C_24H:B14,ASG002_BC3C_24H:B15,ASG002_BC3C_24H:C13,...,SMAD3in_V11,SMAD3in_V12,SMAD3in_V13,SMAD3in_V14,SMAD3in_V15,SMAD3in_V16,SMAD3in_V17,SMAD3in_V18,SMAD3in_V19,SMAD3in_V20
Androgen,-0.052095,-0.009089,0.002049,-0.039778,0.001388,0.020524,0.004034,0.016122,0.021111,0.035960,...,0.012764,0.004873,-0.002185,0.001210,0.000265,0.007804,-0.004034,0.008464,0.003308,0.021534
CDK1_2,-0.768273,-0.614755,0.075325,-0.079904,0.089680,-0.282097,0.017335,-0.287704,0.120143,0.055118,...,-0.480374,-0.437256,-0.400662,-0.417611,-0.386523,-0.403037,-0.394879,-0.437924,-0.435277,-0.472746
CDK4_6,-0.092411,-0.210950,-0.265290,-0.141717,-0.163803,-0.103093,-0.016878,-0.028821,-0.218272,-0.010762,...,0.198343,0.189244,0.177556,0.140909,0.138950,0.150663,0.172020,0.164796,0.190069,0.163447
EGFR,0.598441,0.498146,0.228981,0.371224,0.486402,0.113873,-0.415507,0.263087,0.286899,-0.047077,...,-1.225406,-0.821121,-0.806630,-0.706686,-0.677071,-0.692529,-0.744456,-0.899220,-0.879137,-0.933293
Estrogen,-0.125409,-0.208362,-0.209045,-0.410515,-0.943351,-0.311418,-0.084928,-0.236092,-0.162215,-0.039650,...,-0.296804,-0.265812,-0.238593,-0.263146,-0.243248,-0.251675,-0.229277,-0.276421,-0.260162,-0.304476
FGFR,-0.110612,-0.176118,-0.089455,0.051478,-0.028281,-0.407428,-0.033354,-0.026855,-0.061340,-0.301084,...,-0.854376,-0.700109,-0.638602,-0.638719,-0.569303,-0.617494,-0.624983,-0.711393,-0.717033,-0.792202
PI3K,-1.910966,-1.699159,-1.425575,-1.241725,-0.701968,0.295109,-0.143859,-0.177113,-0.848384,-0.273685,...,-0.177298,-0.139084,-0.120742,-0.114059,-0.099701,-0.140369,-0.131804,-0.135862,-0.155846,-0.180301
p53,-0.210364,-0.217898,-0.125995,-0.404942,0.049696,-1.630029,-1.476804,-0.119178,-0.087987,-1.326474,...,0.016668,-0.000954,0.011649,0.021806,-0.006903,0.014973,0.042119,-0.003211,0.037732,0.001231
TOP2A,-0.194672,0.095617,-0.233097,-0.163237,-0.127492,0.043017,0.079112,-2.000421,-0.196845,-0.298854,...,0.108585,0.092367,0.083997,0.071913,0.067490,0.079518,0.083257,0.086786,0.095575,0.097503
Src,-0.928588,-1.677513,0.522790,-1.205287,0.571455,-1.127273,0.495578,0.394501,0.397006,0.447192,...,-0.127760,-0.154060,-0.098183,-0.127832,-0.056520,-0.117954,-0.098885,-0.113815,-0.155771,-0.166602
